In [ ]:
"""
David - Model 3: Apriori / Frequent Pattern Mining
CSCI 5502 Group 16 — Milestone 3

Requires: mlxtend  (pip install mlxtend)
Fallback:  if mlxtend unavailable, uses built-in manual Apriori
"""

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from google.cloud import bigquery
from itertools import combinations
from collections import defaultdict

import warnings
warnings.filterwarnings("ignore")

try:
    from mlxtend.frequent_patterns import apriori, association_rules
    USE_MLXTEND = True
    print("Using mlxtend Apriori")
except ImportError:
    USE_MLXTEND = False
    print("mlxtend not found — using manual Apriori implementation")

# =========================
# LOAD DATA
# =========================

key_path = "../../../fraud-detection-key.json"

if os.path.exists(key_path):
    client = bigquery.Client.from_service_account_json(key_path)
    query = "SELECT * FROM `csci-4022.CSCI.fraud_data`"
    print("Fetching dataset...")
    df = client.query(query).to_dataframe()
    print(f"Loaded {len(df):,} rows.")
else:
    raise FileNotFoundError("BigQuery key not found at: " + key_path)

# =========================
# WHY APRIORI / FPM?
# =========================
# Unlike Decision Tree and Naïve Bayes, Apriori is NOT a predictive model.
# Its value is EXPLORATORY and EXPLANATORY:
#   - It discovers WHICH combinations of customer behaviors co-occur in fraud
#   - Rules like "high credit risk + foreign request → fraud" give fraud analysts
#     actionable, human-readable patterns to build rule-based alert systems
#   - It surfaces interactions between features that classifiers use implicitly
#     but cannot articulate (e.g. DT splits; NB ignores feature interactions)
#
# Support:    fraction of transactions containing the itemset
# Confidence: P(consequent | antecedent) — how reliable the rule is
# Lift:       how much more likely than random chance — lift > 1 = useful rule

# =========================
# PREPROCESSING FOR FPM
# =========================
# Apriori requires DISCRETE (binary/categorical) items.
# All continuous features must be binned.

print("\n--- DISCRETIZING FEATURES FOR PATTERN MINING ---")

sentinel_cols = ["prev_address_months_count", "bank_months_count"]
for col in sentinel_cols:
    df[col] = df[col].replace(-1, np.nan)

selected_features = [
    "income",
    "credit_risk_score",
    "foreign_request",
    "phone_home_valid",
    "phone_mobile_valid",
    "has_other_cards",
    "email_is_free",
    "keep_alive_session",
    "velocity_6h",
    "velocity_24h",
]

df_ap = df[selected_features + ["fraud_bool"]].copy()

# Fill missing values before binning
df_ap["income"]           = df_ap["income"].fillna(df_ap["income"].median())
df_ap["credit_risk_score"] = df_ap["credit_risk_score"].fillna(df_ap["credit_risk_score"].median())
df_ap["velocity_6h"]      = df_ap["velocity_6h"].fillna(df_ap["velocity_6h"].median())
df_ap["velocity_24h"]     = df_ap["velocity_24h"].fillna(df_ap["velocity_24h"].median())

# Bin continuous features into categories
print("\nBefore discretization:")
print(f"  income range:            [{df_ap['income'].min():.4f}, {df_ap['income'].max():.4f}]")
print(f"  credit_risk_score range: [{df_ap['credit_risk_score'].min()}, {df_ap['credit_risk_score'].max()}]")
print(f"  velocity_6h range:       [{df_ap['velocity_6h'].min():.2f}, {df_ap['velocity_6h'].max():.2f}]")

df_ap["income_bin"]     = pd.qcut(df_ap["income"],     3, labels=["low_inc",  "mid_inc",  "high_inc"])
df_ap["credit_bin"]     = pd.qcut(df_ap["credit_risk_score"], 3, labels=["low_risk", "mid_risk", "high_risk"])
df_ap["velocity6h_bin"] = pd.qcut(df_ap["velocity_6h"].rank(method="first"), 3,
                                   labels=["low_vel6h", "mid_vel6h", "high_vel6h"])
df_ap["velocity24h_bin"]= pd.qcut(df_ap["velocity_24h"].rank(method="first"), 3,
                                   labels=["low_vel24h","mid_vel24h","high_vel24h"])

print("After discretization:")
print(f"  income_bin categories:     {df_ap['income_bin'].unique().tolist()}")
print(f"  credit_bin categories:     {df_ap['credit_bin'].unique().tolist()}")
print(f"  velocity6h_bin categories: {df_ap['velocity6h_bin'].unique().tolist()}")

# Binary/categorical features — keep as-is (already 0/1 or small categories)
binary_features = ["foreign_request", "phone_home_valid", "phone_mobile_valid",
                   "has_other_cards", "email_is_free", "keep_alive_session"]

# Build final item columns
item_cols = ["income_bin", "credit_bin", "velocity6h_bin", "velocity24h_bin"] + binary_features
df_ap = df_ap[item_cols + ["fraud_bool"]].copy()

# Add fraud label as an item
df_ap["fraud_label"] = df_ap["fraud_bool"].map({1: "IS_FRAUD", 0: "IS_LEGIT"})

print(f"\nFraud transactions: {(df_ap['fraud_bool']==1).sum():,}")
print(f"Legit transactions: {(df_ap['fraud_bool']==0).sum():,}")

# =========================
# APRIORI — FRAUD CLASS ONLY
# =========================
# Mining within the fraud class reveals fraud-specific signatures.
# We then compare to legit patterns to find what's exclusive to fraud.

df_fraud = df_ap[df_ap["fraud_bool"] == 1][item_cols].copy()
df_legit = df_ap[df_ap["fraud_bool"] == 0][item_cols].copy().sample(n=min(50000, (df_ap["fraud_bool"]==0).sum()), random_state=42)

print(f"\nMining fraud subset ({len(df_fraud):,} rows)...")
print(f"Mining legit sample ({len(df_legit):,} rows) for comparison...")

def run_mlxtend_apriori(dataframe, min_support, label):
    """Use mlxtend apriori on a one-hot encoded dataframe."""
    # One-hot encode all item columns
    encoded = pd.get_dummies(dataframe.astype(str), prefix_sep="=")
    encoded = encoded.astype(bool)
    frequent = apriori(encoded, min_support=min_support, use_colnames=True, max_len=3)
    if frequent.empty:
        print(f"  [{label}] No frequent itemsets at min_support={min_support}")
        return pd.DataFrame(), pd.DataFrame()
    rules = association_rules(frequent, metric="lift", min_threshold=1.2)
    return frequent, rules


# Manual Apriori (fallback)
def get_frequent_manual(transactions, min_support):
    n = len(transactions)
    min_count = min_support * n
    item_counts = defaultdict(int)
    for t in transactions:
        for item in t:
            item_counts[frozenset([item])] += 1
    frequent = {k: v for k, v in item_counts.items() if v >= min_count}
    all_freq = dict(frequent)
    current = list(frequent.keys())
    k = 2
    while current:
        items = sorted({i for fs in current for i in fs})
        cands = {frozenset(c) for c in combinations(items, k)}
        counts = defaultdict(int)
        for t in transactions:
            ts = set(t)
            for c in cands:
                if c.issubset(ts):
                    counts[c] += 1
        freq_k = {k2: v for k2, v in counts.items() if v >= min_count}
        if not freq_k:
            break
        all_freq.update(freq_k)
        current = list(freq_k.keys())
        k += 1
    return all_freq, n


def get_rules_manual(frequent_itemsets, n, min_conf=0.5, min_lift=1.2):
    rules = []
    for itemset, count in frequent_itemsets.items():
        if len(itemset) < 2:
            continue
        support = count / n
        for i in range(1, len(itemset)):
            for ant in combinations(sorted(itemset), i):
                ant = frozenset(ant)
                con = itemset - ant
                ant_c = frequent_itemsets.get(ant, 0)
                con_c = frequent_itemsets.get(con, 0)
                if ant_c == 0 or con_c == 0:
                    continue
                conf = count / ant_c
                lift = conf / (con_c / n)
                if conf >= min_conf and lift >= min_lift:
                    rules.append({
                        "antecedents": set(ant),
                        "consequents": set(con),
                        "support":    round(support, 6),
                        "confidence": round(conf, 4),
                        "lift":       round(lift, 4),
                    })
    return sorted(rules, key=lambda x: x["lift"], reverse=True)


MIN_SUPPORT    = 0.15   # 15% of the fraud class (lower than typical because fraud class is small)
MIN_CONFIDENCE = 0.50
MIN_LIFT       = 1.2

print(f"\nApriori thresholds: support≥{MIN_SUPPORT}, confidence≥{MIN_CONFIDENCE}, lift≥{MIN_LIFT}")
print("Note: Thresholds relative to each class subset, not full dataset")

if USE_MLXTEND:
    print("\n[mlxtend] Mining fraud patterns...")
    freq_fraud, rules_fraud = run_mlxtend_apriori(df_fraud, MIN_SUPPORT, "FRAUD")
    print(f"  Frequent itemsets (fraud): {len(freq_fraud)}")
    print(f"  Association rules (fraud): {len(rules_fraud)}")

    print("\n[mlxtend] Mining legit patterns...")
    freq_legit, rules_legit = run_mlxtend_apriori(df_legit, MIN_SUPPORT, "LEGIT")
    print(f"  Frequent itemsets (legit): {len(freq_legit)}")

    print("\n=== TOP FRAUD RULES (by lift) ===")
    if not rules_fraud.empty:
        top_rules = rules_fraud.sort_values("lift", ascending=False).head(15)
        print(top_rules[["antecedents", "consequents", "support", "confidence", "lift"]].to_string(index=False))
    else:
        print("No rules found — try lowering min_support")

else:
    # Manual fallback
    def df_to_transactions(dataframe):
        return [
            [f"{col}={row[col]}" for col in dataframe.columns if pd.notna(row[col])]
            for _, row in dataframe.iterrows()
        ]

    print("\n[manual] Converting to transaction format...")
    fraud_transactions = df_to_transactions(df_fraud)
    legit_transactions = df_to_transactions(df_legit)

    print("[manual] Mining fraud patterns...")
    freq_fraud, n_fraud = get_frequent_manual(fraud_transactions, MIN_SUPPORT)
    print(f"  Frequent itemsets (fraud): {len(freq_fraud)}")

    rules_fraud = get_rules_manual(freq_fraud, n_fraud, MIN_CONFIDENCE, MIN_LIFT)
    print(f"  Rules generated (fraud):   {len(rules_fraud)}")

    print("\n[manual] Mining legit patterns...")
    freq_legit, n_legit = get_frequent_manual(legit_transactions, MIN_SUPPORT)
    print(f"  Frequent itemsets (legit): {len(freq_legit)}")

    print("\n=== TOP 15 FRAUD RULES (by lift) ===")
    print(f"{'Antecedent':<45} {'Consequent':<25} {'Supp':>6} {'Conf':>6} {'Lift':>6}")
    print("-" * 92)
    for rule in rules_fraud[:15]:
        ant = ", ".join(sorted(rule["antecedents"]))[:43]
        con = ", ".join(sorted(rule["consequents"]))[:23]
        print(f"{ant:<45} {con:<25} {rule['support']:>6.4f} {rule['confidence']:>6.4f} {rule['lift']:>6.4f}")

# =========================
# COMPARE FRAUD vs LEGIT RULES
# =========================

print("\n=== FRAUD vs LEGIT PATTERN COMPARISON ===")
print("This identifies which patterns are fraud-EXCLUSIVE")
print("(appear frequently in fraud transactions but not legit)")

if USE_MLXTEND and not freq_fraud.empty and not freq_legit.empty:
    fraud_items = set()
    for itemset in freq_fraud["itemsets"]:
        for item in itemset:
            fraud_items.add(item)
    legit_items = set()
    for itemset in freq_legit["itemsets"]:
        for item in itemset:
            legit_items.add(item)
    exclusive = fraud_items - legit_items
    print(f"\nItems frequent in FRAUD but not LEGIT: {sorted(exclusive)}")
elif not USE_MLXTEND:
    fraud_items = {item for fs in freq_fraud for item in fs}
    legit_items = {item for fs in freq_legit for item in fs}
    exclusive = fraud_items - legit_items
    print(f"\nItems frequent in FRAUD but not LEGIT: {sorted(exclusive)}")

# =========================
# FULL DATASET — MLXTEND PATH
# =========================

print("\n=== APRIORI ON MIXED SAMPLE (with fraud_label item) ===")
print("(fraud_bool included as an item in transactions)")

sample_size = 5000
df_sample = df_ap.sample(n=sample_size, random_state=42)

if USE_MLXTEND:
    # Include fraud_label as a column item
    sample_cols = item_cols + ["fraud_label"]
    encoded_sample = pd.get_dummies(df_sample[sample_cols].astype(str), prefix_sep="=")
    encoded_sample = encoded_sample.astype(bool)

    freq_sample = apriori(encoded_sample, min_support=0.01, use_colnames=True, max_len=3)
    if not freq_sample.empty:
        rules_sample = association_rules(freq_sample, metric="lift", min_threshold=1.2)
        fraud_rules = rules_sample[
            rules_sample["consequents"].apply(lambda x: any("IS_FRAUD" in str(i) for i in x))
        ]
        print(f"\nFraud-targeted rules found: {len(fraud_rules)}")
        print("\nTop 10 Fraud Rules:")
        print(fraud_rules.sort_values("lift", ascending=False).head(10)[
            ["antecedents", "consequents", "support", "confidence", "lift"]
        ].to_string(index=False))
    else:
        print("No frequent itemsets found at min_support=0.01")

# =========================
# METRICS SUMMARY TABLE
# =========================

print("\n=== PATTERN MINING EVALUATION METRICS ===")
print(f"{'Metric':<20} {'Value':<20} {'Description'}")
print("-" * 70)
print(f"{'min_support':<20} {MIN_SUPPORT:<20.2f} {'Min fraction of class containing itemset'}")
print(f"{'min_confidence':<20} {MIN_CONFIDENCE:<20.2f} {'Min P(consequent | antecedent)'}")
print(f"{'min_lift':<20} {MIN_LIFT:<20.2f} {'Min ratio vs random — >1 means useful rule'}")
if USE_MLXTEND:
    if not rules_fraud.empty:
        best = rules_fraud.sort_values("lift", ascending=False).iloc[0]
        print(f"\nBest rule lift:       {best['lift']:.4f}")
        print(f"Best rule confidence: {best['confidence']:.4f}")
        print(f"Best rule support:    {best['support']:.4f}")
elif rules_fraud:
    print(f"\nBest rule lift:       {rules_fraud[0]['lift']:.4f}")
    print(f"Best rule confidence: {rules_fraud[0]['confidence']:.4f}")
    print(f"Best rule support:    {rules_fraud[0]['support']:.4f}")

# =========================
# CHALLENGES & SOLUTIONS
# =========================
# Challenge: All features are continuous — Apriori needs discrete items
# Solution:  pd.qcut into 3 quantile bins
#            Binary features (phone_valid, foreign_request) kept as-is
#
# Challenge: Fraud class is only 1.1% — very few fraud transactions (~11,000)
# Solution:  Mine fraud class separately so min_support is relative to fraud size
#            (not full dataset where 0.15 support = 0.15 * 11k = 1,650 transactions,
#             easily achievable; vs. 0.15 * 999k = 150k — too strict)
#
# Challenge: PCA-less features — some are correlated (velocity_6h vs velocity_24h)
# Solution:  Both are included; rules with both simply have higher joint support,
#            and lift > 1 ensures they're not spurious
#
# Challenge: Result interpretation — feature names are behavioral codes
# Solution:  Labeled clearly (credit_bin_high_risk, foreign_request=1, etc.)
#            so fraud analysts can act on the rules directly

print("\n✓ Apriori / Pattern Mining (David) complete.")